# HeritAgent — EXP-0001 koşusu (Colab, A100 80 GB)

Bu notebook asıl deneyi çalıştırır (ADR-0007, ADR-0008). **Önce şunlar tamam olmalı:**
1. pilot notebook'u çalıştı ve `pilot-report.json` incelendi;
2. hesap tavanı kararı verildi;
3. Critic onayı (artık benim görevim) verildi.

**Nasıl çalışır:**
- Koşu birkaç oturuma yayılabilir. Her oturumda "Tümünü çalıştır" deyin; aynı `RUN_ID` ile kaldığı yerden devam eder.
- Kaydedilmiş bir bölüm asla yeniden oynanmaz.
- Koşu yerel diskte çalışır ve kopyası her dakika Drive'a alınır (`heritagent/runs/<RUN_ID>/mirror`). Oturum koparsa en son kopyadan devam eder. Kopyadan sonra yapılmış birkaç dakikalık iş yeniden oynanır.
- `STOP_AFTER_HOURS`: bu oturumda en fazla kaç saat çalışılacağı. Süre dolunca yeni bölüm başlatılmaz; süren bölümler bitip kaydedilir ve hücre temiz durur. CU bütçenizi böyle bölebilirsiniz.
- 96 saatlik tavan yalnızca çalışılan süreyi sayar (ADR-0008 D2). 7000 bölüm ve 225 bin çağrı tavanları da geçerlidir; tavanda koşu "incomplete" olarak durur.
- **Aynı koşuyu asla iki sekmede ya da iki oturumda birden çalıştırmayın.** İkincisi birincisini durdurur.

**Tahmini maliyet (M):** pilot projeksiyonuna bakın. Örneğin saatte 500 bölümle 5000 bölüm yaklaşık 10 saat eder, saatte 7.5 CU ile yaklaşık 75 CU.

In [ ]:
# Ayarlar
RUN_ID = "EXP-0001-run-1"  # yeni bir koşu için yeni ad; devam için aynı ad
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # yalnızca YENİ koşuda; devamda koşunun kendi commit'i kullanılır
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # pilotla aynı değer önerilir
STOP_AFTER_HOURS = 6.0     # bu oturumda en fazla bu kadar saat; None = bitene kadar
BATCH_INVARIANT = True     # pilotla aynı değer olmalı (ADR-0008 D4)

In [ ]:
# 1) GPU: yalnızca tek bir A100 80 GB (ADR-0008 D1). Başka bir GPU'da hiçbir şey kurulmaz.
import re, subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(query.stdout or query.stderr)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()]
assert len(gpus) == 1 and "A100" in gpus[0][0] and int(float(gpus[0][1])) >= 76000, (
    "Bu notebook yalnızca TEK bir A100 80 GB ile çalışır. Çalışma zamanı > Çalışma zamanı "
    "türünü değiştir > A100 GPU seçin ve yeniden bağlanın.")
header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
CUDA_DRIVER = float(re.search(r"CUDA Version:\s*([0-9.]+)", header).group(1))
print("Sürücünün desteklediği en yüksek CUDA:", CUDA_DRIVER)

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
# 2b) Koşunun kayıtları: commit ve model revizyonu sabitlenir.
import os
RUN_DRIVE = f"{DRIVE_ROOT}/runs/{RUN_ID}"
os.makedirs(RUN_DRIVE, exist_ok=True)
commit_pin = f"{RUN_DRIVE}/code-commit.txt"
RUN_REF = open(commit_pin).read().strip() if os.path.exists(commit_pin) else CODE_REF
revision_file = f"{DRIVE_ROOT}/model-revision.txt"
assert os.path.exists(revision_file), "Önce pilot notebook'unu çalıştırın: model revizyonu orada sabitlenir."
MODEL_REVISION = open(revision_file).read().strip()
print("Kod:", RUN_REF, "| model revizyonu:", MODEL_REVISION)

In [ ]:
# 3) Kod: depoyu klonla ve RUN_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)
ref = RUN_REF
target = f"origin/{ref}" if subprocess.run(
    ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
    capture_output=True).returncode == 0 else ref
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                        text=True, check=True).stdout.strip()
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 3b) Yeni koşuda commit Drive'a yazılır; devam eden koşu hep aynı commit'le çalışır.
if not os.path.exists(commit_pin):
    open(commit_pin, "w").write(COMMIT)
assert open(commit_pin).read().strip() == COMMIT, "Kod commit'i koşunun commit'i değil."

In [ ]:
# 4) Kurulum (~5-10 dk): vLLM 0.29.0 + torch 2.13.0, ALFWorld 0.4.2, TextWorld 1.7.0.
# Sürücü CUDA 13'ü destekliyorsa CUDA 13 paketi, desteklemiyorsa resmi CUDA 12.9 paketi.
import subprocess, sys
heritagent = f"{REPO}[alfworld]"
if CUDA_DRIVER >= 13.0:
    command = ["vllm==0.29.0", "-e", heritagent]
else:
    command = ["vllm==0.29.0+cu129", "torch==2.13.0+cu129", "torchvision==0.28.0+cu129",
               "torchaudio==2.11.0+cu129", "-e", heritagent,
               "--extra-index-url", "https://wheels.vllm.ai/0.29.0/cu129",
               "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *command], check=True)
print(subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, "
                      "'torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 5) ALFWorld verisi (~2-5 dk). Oyunlar, donmuş manifestteki içerik hash'leriyle tek tek
# doğrulanır; tutmayan bir oyun hiç açılmaz.
import os, subprocess
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
subprocess.run(["alfworld-download"], check=True, env=os.environ)
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
print(sorted(os.listdir(DATA_ROOT)))

In [ ]:
# 6) Aynı A100 kontrolü, deneyin kendi koduyla (fail-closed).
!python -m evaluation.exp0001.cli gpu-check

In [ ]:
# 7) Model: pilotun sabitlediği revizyon (koşu başında kaydedilen ağırlık hash'iyle karşılaştırılır).
!python -m evaluation.exp0001.cli prepare-model --model Qwen/Qwen2.5-7B-Instruct --revision {MODEL_REVISION} --out /content/prepared.json

In [ ]:
# 8) Model sunucusu (~5-15 dk): vllm serve, batch-invariant çekirdekler açık (ADR-0008 D4).
# Hazır olunca "the model server ... serves" yazar; günlük: /content/vllm-server.log
# Sunucu açılmazsa günlüğün sonuna bakın: !tail -50 /content/vllm-server.log
bi_flag = "" if BATCH_INVARIANT else "--no-batch-invariant"
!python -m evaluation.exp0001.cli serve --prepared /content/prepared.json --log /content/vllm-server.log --info /content/vllm-server.json {bi_flag}

In [ ]:
# 8b) Ön kontrol (2 model çağrısı, hiçbir deney kaydına girmez): ReAct biçimli bir yanıt ve
# miner'ın JSON şemasıyla yapılandırılmış bir yanıt. Miner normalde ancak 540 lifetime
# bölümünden sonra ilk kez çalışır; burada hata verirse CU harcamadan önce durursunuz.
!python -m evaluation.exp0001.cli preflight --prepared /content/prepared.json

In [ ]:
# 9) EXP-0001: başlat ya da devam et. Bu hücre saatlerce sürebilir; sekmeyi açık tutun.
# Her 50 bölümde bir ilerleme satırı yazar.
stop = "" if STOP_AFTER_HOURS is None else f"--stop-after-hours {STOP_AFTER_HOURS}"
!python -m evaluation.exp0001.cli run --prepared /content/prepared.json --data-root {DATA_ROOT} --repo /content/heritagent --server-info /content/vllm-server.json --workers {WORKERS} --local-dir /content/exp0001-run --mirror-dir {RUN_DRIVE}/mirror --run-id {RUN_ID} {stop}

In [ ]:
# 10) Durum (yalnızca okur): bölümler, çağrılar, çalışılan saat, oturumlar.
!python -m evaluation.exp0001.cli status --run-dir /content/exp0001-run --mirror-dir {RUN_DRIVE}/mirror

In [ ]:
# 11) Koşu bittiyse rapor Drive'a kopyalanır.
import json, os, shutil
if os.path.exists("/content/exp0001-run/report.json"):
    shutil.copy("/content/exp0001-run/report.json", f"{RUN_DRIVE}/report.json")
    report = json.load(open("/content/exp0001-run/report.json"))
    print({k: report.get(k) for k in ("status", "stage", "reason", "episodes", "model_calls")})
    print("Rapor:", f"{RUN_DRIVE}/report.json", "— bu dosyayı gönderin.")
else:
    print("Koşu henüz bitmedi; sonraki oturumda aynı RUN_ID ile 'Tümünü çalıştır' deyin.")

**Bitti mi?** `report.json` dosyasını gönderin. Final istatistik koşu sonrası Critic incelemesinden geçer; hiçbir sonuç "doğrulandı" diye etiketlenmez (SPEC §32).

Oturumu kapatmak için: Çalışma zamanı > **Bağlantıyı kes ve çalışma zamanını sil**.